# Cortex Agent Streaming Demo

This notebook demonstrates how to call the **Snowflake Cortex Agent REST API**
directly with **SSE streaming**, parse the event stream in pure Python, and display
rich results (text, SQL, tables, charts).

1. **Setup** — imports, session, JWT
2. **Rich display** — `display_result()` with thinking, badges, SQL, DataFrames, charts
3. **One-shot** — `run_agent()` for single questions
4. **Multi-turn** — `AgentChat` with local history
5. **Thread mode** — server-side continuity via `create_thread()`
6. **Quiet collection** — `collect_agent_events()` for pipelines
7. **Custom streaming** — `iter_normalized_agent_events()` for custom UIs
8. **Raw event inspector** — see every SSE event from the wire

## 1. Setup and Configuration

In [ ]:
#| eval: false
import json
import pandas as pd
from IPython.display import display, Markdown, HTML

from mcp_ski_resort.core import (
    default_session, get_headers,
    stream_agent_sse, normalize_event, AgentResult,
    run_agent, collect_agent_events, iter_normalized_agent_events,
    result_set_to_dataframe,
    AgentChat, create_thread,
)

session = default_session()
token = session.jwt_gen.get_token()
print(f"Account: {session.account}")
print(f"Host:    {session.host}")
print(f"JWT:     {token[:40]}...OK")

## 2. Rich Display Helper

Renders the full `AgentResult` with collapsible thinking, tool-use badges,
Markdown answer, syntax-highlighted SQL, DataFrames, Altair charts, errors,
and a status bar.

In [ ]:
#| eval: false
def display_result(result: AgentResult):

    if result.thinking:
        thinking_text = "\n\n".join(result.thinking)
        display(HTML(
            f'<details style="margin-bottom:12px;">'
            f'<summary style="cursor:pointer;font-weight:600;color:#6b7280;">'
            f'Thinking ({len(result.thinking)} steps)</summary>'
            f'<div style="padding:8px 16px;background:#f9fafb;border-radius:8px;margin-top:4px;'
            f'font-size:13px;color:#4b5563;white-space:pre-wrap;">{thinking_text[:2000]}</div>'
            f'</details>'
        ))

    if result.tools_used:
        badges = " ".join(
            f'<span style="display:inline-block;padding:2px 8px;margin:2px;background:#eff6ff;'
            f'color:#2563eb;border-radius:9999px;font-size:12px;border:1px solid #bfdbfe;">'
            f'{t}</span>'
            for t in result.tools_used
        )
        display(HTML(f'<div style="margin-bottom:8px;">{badges}</div>'))

    if result.answer:
        display(Markdown(result.answer))

    for i, sql in enumerate(result.sql_queries):
        display(HTML(
            f'<details style="margin:8px 0;">'
            f'<summary style="cursor:pointer;font-weight:600;color:#6b7280;">'
            f'SQL Query {i+1}</summary>'
            f'<pre style="padding:12px;background:#1e293b;color:#e2e8f0;border-radius:8px;'
            f'overflow-x:auto;font-size:13px;">{sql}</pre>'
            f'</details>'
        ))

    for i, df in enumerate(result.dataframes):
        display(HTML(f'<p style="font-weight:600;color:#374151;margin:8px 0 4px;">Result {i+1} ({len(df)} rows)</p>'))
        display(df.head(20))

    for i, spec in enumerate(result.chart_specs):
        try:
            import altair as alt
            chart = alt.Chart.from_dict(spec)
            display(chart)
        except ImportError:
            display(HTML(
                f'<details><summary>Chart Spec {i+1} (install altair to render)</summary>'
                f'<pre>{json.dumps(spec, indent=2)[:2000]}</pre></details>'
            ))
        except Exception:
            display(HTML(
                f'<details><summary>Chart Spec {i+1} (raw JSON)</summary>'
                f'<pre>{json.dumps(spec, indent=2)[:2000]}</pre></details>'
            ))

    if result.errors:
        for err in result.errors:
            display(HTML(
                f'<div style="padding:12px;background:#fef2f2;border:1px solid #fecaca;'
                f'border-radius:8px;color:#991b1b;">{err}</div>'
            ))

    display(HTML(
        f'<div style="margin-top:12px;padding:8px 12px;background:#f0fdf4;border-radius:8px;'
        f'font-size:12px;color:#166534;">'
        f'Completed in {result.duration_seconds}s | '
        f'Tools: {len(result.tools_used)} | '
        f'SQL queries: {len(result.sql_queries)} | '
        f'Result sets: {len(result.dataframes)} | '
        f'Charts: {len(result.chart_specs)}'
        f'</div>'
    ))

---
## 3. Demo: Resort Executive Agent

The `RESORT_EXECUTIVE` agent has access to all 11 semantic views and can
synthesize across domains.

In [ ]:
#| eval: false
result = run_agent(
    agent_name="RESORT_EXECUTIVE",
    question="Give me a complete resort performance summary for the 2024-2025 season",
)
display_result(result)

## 4. Demo: Ski Ops Assistant

The `SKI_OPS_ASSISTANT` focuses on lift operations, staffing, weather, and safety.

In [ ]:
#| eval: false
result = run_agent(
    agent_name="SKI_OPS_ASSISTANT",
    question="What are the average wait times by lift on weekends for the 2024-2025 season?",
)
display_result(result)

## 5. Multi-Turn Conversation with AgentChat

`AgentChat` manages conversation history automatically — no manual tracking
needed. For lower-level control, use `run_agent()` or `stream_agent_sse()` directly.

In [ ]:
#| eval: false
chat = AgentChat("RESORT_EXECUTIVE")

r1 = chat.ask("How does weather impact our daily revenue?")
display_result(r1)
print(f"\n{chat}")

In [ ]:
#| eval: false
r2 = chat.ask("Now break that down by powder days vs non-powder days")
display_result(r2)
print(f"\n{chat}")
print(f"History entries: {len(chat.history)}")

## 6. Thread Mode (Server-Side Continuity)

Cortex Threads let the server own conversation continuity. History is
recorded locally for inspection but **not** sent upstream — the `thread_id`
and `parent_message_id` handle it.

In [ ]:
#| eval: false
tid = create_thread()
print(f"Thread: {tid}")

chat = AgentChat("RESORT_EXECUTIVE", thread_id=tid)

r1 = chat.ask("What is our revenue trend this season?")
display_result(r1)

r2 = chat.ask("Break that down by month.")
display_result(r2)

print(f"\nParent message ID: {chat._parent_message_id}")
print(f"Thread metadata: {chat.last.thread_metadata}")
print(chat)

## 7. Quiet Collection (No Printing)

`collect_agent_events()` accumulates the full `AgentResult` with zero
side effects — useful for pipelines and batch scripts.

In [ ]:
#| eval: false
result = collect_agent_events(
    "RESORT_EXECUTIVE",
    "Give me a one-paragraph executive summary of this season.",
)
display_result(result)

## 8. Custom Streaming

`iter_normalized_agent_events()` yields normalized `{"event": str, "data": dict}`
dicts — ideal for building custom UIs or SSE proxies.

In [ ]:
#| eval: false
for evt in iter_normalized_agent_events(
    "RESORT_EXECUTIVE", "How many trails are currently open?"
):
    etype = evt["event"]
    if etype == "text":
        print(evt["data"]["text"], end="", flush=True)
    elif etype == "tool_use":
        print(f"\n[Tool: {evt['data']['name']}]")
    elif etype == "sql":
        print(f"[SQL: {evt['data']['statement'][:80]}...]")
    elif etype == "table":
        print(f"[Table: {len(evt['data'].get('data', []))} rows]")
print()

## 9. Raw Event Inspector

For debugging: see every raw SSE event from the agent.

In [ ]:
#| eval: false
print("Raw SSE events from RESORT_EXECUTIVE:")
print("=" * 60)

for i, raw in enumerate(stream_agent_sse(
    "RESORT_EXECUTIVE",
    "How many total visits did we have last season?"
)):
    evt = raw["event"]
    data_preview = json.dumps(raw.get("data", {}))[:200]
    print(f"[{i:03d}] event={evt:<30s} data={data_preview}")
    if evt == "done":
        break

print("=" * 60)
print("Stream complete")